# Logistic Regression - Sweeps

## Setup and Imports

In [1]:
import sys

sys.path.append("..")

import dotenv

import wandb
from src.api.run import sweep_logistic_regression
from src.api.sweep import wandb_sweep

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: v8-luky (aicomp-mmlm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Experiments

In [3]:
max_runs = 100
sweep_config = {
    "name": "Logistic Regression",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "logistic_regression_config": {
            "parameters": {
                "penalty": {"values": ["l1", "l2", "elasticnet", None]},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "solver": {"values": ["lbfgs", "liblinear", "saga"]},
                "tol": {"min": 1e-5, "max": 1e-3},
                "max_iter": {"value": 1000},
                "l1_ratio": {"distribution": "uniform", "min": 0.0, "max": 1.0},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_logistic_regression, run_count=max_runs, project="logistic-regression")

## Submission from Best Model

In [5]:
from src.dataloaders.simple import SeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.experiments.config import RunConfig
from src.models.logistic_regression import LogisticRegressionHyperparamConfig, LogisticRegressionModel
from src.submissions import create_submission

In [ ]:
run = wandb.Api().run("j7xmtprd")
config = run.config
config

{'run_config': {'num_features': 5, 'start_season': 2003, 'valid_season': 2024},
 'logistic_regression_config': {'C': 34.53696852319429,
  'tol': 0.0005380404159075536,
  'solver': 'saga',
  'penalty': None,
  'l1_ratio': 0.5184831647240584,
  'max_iter': 1000}}

In [7]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = SeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=5, valid_season=2024, start_season=2003, data_loader='season_average')

In [8]:
hyperparameters = LogisticRegressionHyperparamConfig(**config.get("logistic_regression_config", {}))
hyperparameters

LogisticRegressionHyperparamConfig(penalty=None, dual=False, tol=0.0005380404159075536, C=34.53696852319429, fit_intercept=True, intercept_scaling=1.0, class_weight=None, random_state=42, solver='saga', max_iter=1000, verbose=0, warm_start=False, n_jobs=-1, l1_ratio=0.5184831647240584)

In [9]:
model = LogisticRegressionModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [ ]:
season = 2025
create_submission(season=season, model=model, filename=f"submission_logistic_regression_{season}.csv", fit=True)

## Sweep with Default Features

In [3]:
max_runs = 100
sweep_config = {
    "name": "Logistic Regression (Default Features)",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "logistic_regression_config": {
            "parameters": {
                "penalty": {"values": ["l1", "l2", "elasticnet", None]},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "solver": {"values": ["lbfgs", "liblinear", "saga"]},
                "tol": {"min": 1e-5, "max": 1e-3},
                "max_iter": {"value": 1000},
                "l1_ratio": {"distribution": "uniform", "min": 0.0, "max": 1.0},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"value": 0},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_logistic_regression, run_count=max_runs, project="logistic-regression")

## Submission from Best Model

In [6]:
from src.dataloaders.simple import SeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.experiments.config import RunConfig
from src.models.logistic_regression import LogisticRegressionHyperparamConfig, LogisticRegressionModel
from src.submissions import create_submission

In [7]:
run = wandb.Api().run("rbe4vgv4")
config = run.config
config

{'run_config': {'num_features': 0, 'start_season': 2003, 'valid_season': 2024},
 'logistic_regression_config': {'C': 0.022807709448714443,
  'tol': 0.0006910400762252707,
  'solver': 'saga',
  'penalty': 'l1',
  'l1_ratio': 0.7450550437712368,
  'max_iter': 1000}}

In [8]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = SeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=0, valid_season=2024, start_season=2003, data_loader='season_average')

In [9]:
hyperparameters = LogisticRegressionHyperparamConfig(**config.get("logistic_regression_config", {}))
hyperparameters

LogisticRegressionHyperparamConfig(penalty='l1', dual=False, tol=0.0006910400762252707, C=0.022807709448714443, fit_intercept=True, intercept_scaling=1.0, class_weight=None, random_state=42, solver='saga', max_iter=1000, verbose=0, warm_start=False, n_jobs=-1, l1_ratio=0.7450550437712368)

In [10]:
model = LogisticRegressionModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [11]:
season = 2025
create_submission(
    season=season, model=model, filename=f"submission_logistic_regression_default_features_{season}.csv", fit=True
)

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(


metrics: {'train_brier': np.float64(0.16687414056732564)}, step: None


WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/submission_logistic_regression_default_features_2025.csv')

## Sweep with Weighted Season Average DataLoader

In [3]:
max_runs = 100
sweep_config = {
    "name": "Logistic Regression",
    "method": "bayes",
    "metric": {"name": "valid_brier", "goal": "minimize"},
    "parameters": {
        "logistic_regression_config": {
            "parameters": {
                "penalty": {"values": ["l1", "l2", "elasticnet", None]},
                "C": {"distribution": "log_uniform_values", "min": 1e-4, "max": 1e2},
                "solver": {"values": ["lbfgs", "liblinear", "saga"]},
                "tol": {"min": 1e-5, "max": 1e-3},
                "max_iter": {"value": 1000},
                "l1_ratio": {"distribution": "uniform", "min": 0.0, "max": 1.0},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2000},
                "start_season": {"value": 2000},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
                "data_loader": {"value": "weighted_season_average"},
                "data_loader_config": {
                    "parameters": {
                        "regular_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "tourney_weight": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                        "discount_factor": {"distribution": "uniform", "min": 0.9, "max": 1.0},
                    }
                },
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_logistic_regression, run_count=max_runs, project="logistic-regression")

## Submission from Best Model

In [3]:
from src.dataloaders.simple import SeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.experiments.config import RunConfig
from src.models.logistic_regression import LogisticRegressionHyperparamConfig, LogisticRegressionModel
from src.submissions import create_submission

In [5]:
run = wandb.Api().run("logistic-regression/runs/qw59xwgs")
config = run.config
config

{'run_config': {'data_loader': 'weighted_season_average',
  'num_features': 81,
  'start_season': 2000,
  'valid_season': 2000,
  'data_loader_config': {'regular_weight': 0.6657392361672186,
   'tourney_weight': 0.6046129335992463,
   'discount_factor': 0.9984067716404904}},
 'logistic_regression_config': {'C': 21.647962697783267,
  'tol': 0.00038194186036978514,
  'solver': 'saga',
  'penalty': 'l2',
  'l1_ratio': 0.26605372654305814,
  'max_iter': 1000}}

In [6]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = SeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=81, valid_season=2000, start_season=2000, data_loader='weighted_season_average', data_loader_config={'regular_weight': 0.6657392361672186, 'tourney_weight': 0.6046129335992463, 'discount_factor': 0.9984067716404904})

In [7]:
hyperparameters = LogisticRegressionHyperparamConfig(**config.get("logistic_regression_config", {}))
hyperparameters

LogisticRegressionHyperparamConfig(penalty='l2', dual=False, tol=0.00038194186036978514, C=21.647962697783267, fit_intercept=True, intercept_scaling=1.0, class_weight=None, random_state=42, solver='saga', max_iter=1000, verbose=0, warm_start=False, n_jobs=-1, l1_ratio=0.26605372654305814)

In [8]:
model = LogisticRegressionModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [9]:
season = 2025
create_submission(season=season, model=model, filename=f"logistic_regression_weighted_avg_{season}.csv", fit=True)

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


metrics: {'train_brier': np.float64(0.1632123426834758)}, step: None


WindowsPath('C:/Users/kybur/Repos/HSLU/aicomp/code/submissions/logistic_regression_weighted_avg_2025.csv')